# Topic: Vulnerability Scanner Simulator - Banner audits, CVE database lookups, and security risk reports
 
## 1. THE CONCEPTS OF VULNERABILITY SCANNING
- **Vulnerability Scanner**: An automated tool that sweeps targets to identify security weaknesses (vulnerabilities).
- **Non-Intrusive Scanning**:
  - Scanners connect to services to read version banners without delivering payloads or exploit chains.
  - The extracted banner version is matched against a database of **Common Vulnerabilities and Exposures (CVE)**.
  - **CVE**: A standardized dictionary cataloging publicly disclosed cybersecurity vulnerabilities (managed by MITRE corporation).
 
## 2. MAPPING BANNERS TO CVEs
- Real scanners query remote servers or local databases (like the National Vulnerability Database - NVD) to check if a specific software version is marked vulnerable.
- In this lab, we build a **Vulnerability Scanner Simulator** that maps extracted service banners to a mock local CVE dictionary, demonstrating this correlation.
 
---


In [ ]:
import re

class VulnerabilityScanner:
    # Mock CVE database mapping software banners to known CVE identifiers
    MOCK_CVE_DATABASE = {
        r"ssh-2.0-openssh_7.2": [
            {"cve": "CVE-2016-6210", "severity": "Medium", "desc": "User enumeration vulnerability via timing differences."},
            {"cve": "CVE-2018-15473", "severity": "Medium", "desc": "Username enumeration vulnerability."}
        ],
        r"vsftpd_2.3.4": [
            {"cve": "CVE-2011-2523", "severity": "Critical", "desc": "Backdoor command execution in vsftpd 2.3.4 package download."}
        ],
        r"apache/2.4.41": [
            {"cve": "CVE-2020-1927", "severity": "High", "desc": "Mod_rewrite open redirect vulnerability."}
        ]
    }

    def __init__(self):
        pass

    def check_banner_for_cves(self, banner):
        """Matches a banner against our mock CVE database using regex."""
        banner_lower = banner.lower()
        matched_cves = []
        
        for signature, cves in self.MOCK_CVE_DATABASE.items():
            if re.search(signature, banner_lower):
                matched_cves.extend(cves)
                
        return matched_cves

    def run_vulnerability_audit(self, target_host, scan_results):
        """Generates a vulnerability audit report from scan results (ip, port, banner)."""
        print(f"--- Starting Vulnerability Audit Report for {target_host} ---")
        vulnerabilities_found = 0
        
        for port, banner in scan_results:
            if not banner:
                continue
                
            cves = self.check_banner_for_cves(banner)
            if cves:
                print(f"\n[!] WARNING: Vulnerable Service on Port {port}!")
                print(f"    Service Banner: {banner}")
                for entry in cves:
                    print(f"    * {entry['cve']} [{entry['severity']}] - {entry['desc']}")
                    vulnerabilities_found += 1
            else:
                # Version either secure or not in database
                print(f"\n[+] Port {port}: Service looks secure or has no matching CVEs in database.")
                
        print(f"\n--- Audit Finished. Total CVEs identified: {vulnerabilities_found} ---")



In [ ]:
# Simulating the scan pipeline
print("--- Simulating Active Banner Audit ---")

# Mock results returned by a port scanner: list of tuples (port, banner)
mock_port_scan_results = [
    (21, "vsFTPd 2.3.4"),                                 # Vulnerable critical
    (22, "SSH-2.0-OpenSSH_7.2p2 Ubuntu-4ubuntu2.8"),      # Vulnerable medium
    (80, "Apache/2.4.41 (Ubuntu)"),                       # Vulnerable high
    (443, "SSH-2.0-OpenSSH_8.9p1")                        # Secure / Not matched
]

scanner = VulnerabilityScanner()
scanner.run_vulnerability_audit("192.168.1.100", mock_port_scan_results)
